# Intruction
Following [`preprocessing.ipynb`](/notebooks/preprocessing.ipynb), we can now begin with testing our
datasets on different models.

Similarly to our *exploratory data analysis* portion, we will focus on a curated set of agencies to
base our initial model selection. We will focus on the **NYPD**, **DOT**, and **TLC**.

We are again, attempting to determine **resolution time**. This is a regression task. Observing our
EDA, we found that our agencies had large outliers in our target variable. As such, we will evaluate 
our models based on **Mean Average Error**. *MAE* gives us a more robust statistic, and is less
sensitive to outliers than other statistics, like *RMSE*. We can also use metrics like **symmetric 
mean absolute percentage error** (*sMAPE*) and **R-squared** to evaluate our models across the different
scales of `resolution_time`.

# Setup
Here, we will establish our three agencies, as well as create our `X` and `y` features. Since each 
agency contains different magnitudes and fields, we will implement our previously created 
preprocessing pipeline *per-agency*.

In [143]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# grab data
df = pd.read_csv(
    '../data/nyc311_2025.csv', 
    index_col='id', 
    dtype={'zipcode': 'category'},
    parse_dates=['created', 'closed']
)

In [144]:
df.info()

<class 'pandas.DataFrame'>
Index: 3475290 entries, 67351762 to 63577994
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   created          datetime64[us]
 1   closed           datetime64[us]
 2   agency_name      str           
 3   problem          str           
 4   detail           str           
 5   borough          str           
 6   lat              float64       
 7   long             float64       
 8   method           str           
 9   zipcode          category      
 10  resolution_time  float64       
dtypes: category(1), datetime64[us](2), float64(3), str(5)
memory usage: 298.3 MB


In [145]:
# initialize agencies

nypd = df[df.agency_name == 'New York City Police Department']
dot = df[df.agency_name == 'Department of Transportation']
tlc = df[df.agency_name == 'Taxi and Limousine Commission']

pd.concat([nypd.head(2), dot.head(2), tlc.head(2)])

,created,closed,agency_name,problem,detail,borough,lat,long,method,zipcode,resolution_time
id,,,,,,,,,,,
67351762,2025-12-31 23:59:28,2026-01-01 00:40:32,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.792141,-73.950097,MOBILE,10029,0.684
67344624,2025-12-31 23:59:23,2026-01-01 01:03:42,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.825137,-73.949447,ONLINE,10031,1.072
67344470,2025-12-31 23:57:00,2026-01-09 01:53:00,Department of Transportation,Street Light Condition,Street Light Out,BRONX,40.871462,-73.830537,UNKNOWN,10475,193.933
67348068,2025-12-31 23:57:00,2026-01-07 09:09:00,Department of Transportation,Street Light Condition,Street Light Out,BRONX,40.861213,-73.825111,UNKNOWN,10475,153.200
67345192,2025-12-31 23:58:25,2026-01-02 11:55:46,Taxi and Limousine Commission,Lost Property,Bag/Wallet,MANHATTAN,40.738077,-73.992123,PHONE,10011,35.956
67348740,2025-12-31 23:20:23,2026-01-02 13:27:51,Taxi and Limousine Commission,Lost Property,Bag/Wallet,QUEENS,40.648320,-73.788281,ONLINE,11430,38.124


In [146]:
agencies = {
    'nypd': {
        'df': nypd
    },
    'dot': {
        'df': dot
    },
    'tlc': {
        'df': tlc
    }
}

## Pipeline import

In [147]:
import cloudpickle as cp

preprocessing_pipeline = None
with open('../models/preprocessing_pipeline.pkl', 'rb') as f:
    preprocessing_pipeline = cp.load(f)

## Train-test split

In [148]:
from sklearn.model_selection import train_test_split

def populate_train_test(agencies: dict):
    '''
    Will give each agency in the agency dict proper X_train, X_test, y_train, y_test splits usign
    `sklearn.model_selection.train_test_split()`
    '''
    
    X = ['created', 'closed', 'detail', 'borough', 'method', 'problem', 'zipcode', 'lat', 'long'] # list of all used features
    y = 'resolution_time'
    for a in agencies.keys():
        curr_df = agencies[a]['df']
        X_train, X_test, y_train, y_test = train_test_split(curr_df[X], curr_df[y], random_state=42, train_size=0.8)
        # populate
        agencies[a]['X_train'] = X_train
        agencies[a]['X_test'] = X_test
        agencies[a]['y_train'] = y_train
        agencies[a]['y_test'] = y_test

In [149]:
populate_train_test(agencies)

In [150]:
for k in agencies.keys():
    print(k, len(agencies[k]['X_train']), len(agencies[k]['X_test']))

nypd 1361864 340466
dot 138912 34729
tlc 22430 5608


# Model Selection
Next, we will begin model selection.

Our process begins with simply creating a pipeline which contains both our `preprocessing_pipeline`
along with the model of interest. 

As stated previouly, we are going to evaluate our models using
**MAE**, **R^2**, and **sMAPE**. Along with using *cross-validation*, with `cv=5`.

We are going to evaluate the following models:
- `Ridge`: A simple linear model to use with regularization
- `SVR`: A Support Vector Machine that can be used to perform regresssion. 

In [151]:
# imports

# model testing function 
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVR, SVR
import xgboost

import warnings
# Our pipeline will create an infrequent column if there are rare problems unseen during pipeline fit
# This is intended behavior. We will ignore warning messages to avoid clutter.
warnings.filterwarnings('ignore', message='Found unknown categories.*', category=UserWarning)

In [152]:
def test_model(agencies: dict, model: any, model_name: str, verbose: int=0, n_jobs:int = 4):
    for a in agencies.keys():
        # -- define pipeline
        pipe = Pipeline([
            ('preprocessing', preprocessing_pipeline),
            (model_name, model)
        ])

        # -- cross validate
        results = cross_validate(
                pipe,
                agencies[a]['X_train'],
                agencies[a]['y_train'],
                cv=5,
                scoring='r2',
                n_jobs=n_jobs,
                verbose=verbose,
                return_estimator=True
        )

        # -- calculate statistics
        best_model = results['estimator'][np.argmax(results['test_score'])] # best model
        y_true = agencies[a]['y_train']
        y_pred = best_model.predict(agencies[a]['X_train'])

        r2 = r2_score(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = root_mean_squared_error(y_true, y_pred)

        agencies[a][model_name] = {}
        agencies[a][model_name]['cv_scores'] = results['test_score']
        agencies[a][model_name]['best_r2'] = r2
        agencies[a][model_name]['best_mae'] = mae
        agencies[a][model_name]['best_mape'] = mape
        agencies[a][model_name]['best_rmse'] = rmse

def print_scores(agencies, model_name):
        print(f'Best-performing model scores for {model_name}:')
        for a in agencies.keys():
                res = agencies[a][model_name]

                print(f'''{a}:
        CV R^2:       {res['cv_scores']}                  
                                                                                       
        results of model predictions on all training data:
        BEST MAE:     {res['best_mae']:.2f}
        BEST RMSE:    {res['best_rmse']:.2f}
        BEST MAPE:    {res['best_mape']*100:.2f}
        BEST R^2:     {res['best_r2']:.3f}
''')

In [153]:
test_model(agencies, Ridge(), 'Ridge')
print_scores(agencies, 'Ridge')

Best-performing model scores for Ridge:
nypd:
        CV R^2:       [0.0446468  0.06208572 0.04436307 0.06581192 0.04501242]                  

        results of model predictions on all training data:
        BEST MAE:     2.64
        BEST RMSE:    8.49
        BEST MAPE:    588.96
        BEST R^2:     0.051

dot:
        CV R^2:       [0.14718515 0.14869126 0.13636934 0.13336136 0.15135643]                  

        results of model predictions on all training data:
        BEST MAE:     256.51
        BEST RMSE:    690.59
        BEST MAPE:    5179.12
        BEST R^2:     0.148

tlc:
        CV R^2:       [0.34328987 0.33492107 0.33777337 0.34280473 0.35261757]                  

        results of model predictions on all training data:
        BEST MAE:     1288.86
        BEST RMSE:    1702.80
        BEST MAPE:    7355.02
        BEST R^2:     0.348



In [154]:
test_model(agencies, LinearSVR(), 'LinearSVR')
print_scores(agencies, 'LinearSVR')

Best-performing model scores for LinearSVR:
nypd:
        CV R^2:       [0.00162919 0.00295446 0.0020617  0.00397811 0.00231458]                  

        results of model predictions on all training data:
        BEST MAE:     2.27
        BEST RMSE:    8.70
        BEST MAPE:    282.39
        BEST R^2:     0.002

dot:
        CV R^2:       [0.02516999 0.02230206 0.02059877 0.02155334 0.02435018]                  

        results of model predictions on all training data:
        BEST MAE:     194.37
        BEST RMSE:    739.63
        BEST MAPE:    752.71
        BEST R^2:     0.023

tlc:
        CV R^2:       [0.19086328 0.18935143 0.19325121 0.21803338 0.21145762]                  

        results of model predictions on all training data:
        BEST MAE:     1267.99
        BEST RMSE:    1874.25
        BEST MAPE:    1201.79
        BEST R^2:     0.210



In [155]:
# xgboost

# .22, .22, .49
# .22, .27, .50 - full 
# .16, .27, .54 - no zipcode
# .37, .38, .55 - including zipcode, lat & long
test_model(agencies, xgboost.XGBRegressor(), 'XGBoost')
print_scores(agencies, 'XGBoost')

Best-performing model scores for XGBoost:
nypd:
        CV R^2:       [0.25841203 0.25158895 0.23944926 0.24430839 0.17783579]                  

        results of model predictions on all training data:
        BEST MAE:     2.15
        BEST RMSE:    7.02
        BEST MAPE:    388.36
        BEST R^2:     0.352

dot:
        CV R^2:       [0.18576358 0.19064945 0.13955742 0.16019365 0.1848662 ]                  

        results of model predictions on all training data:
        BEST MAE:     220.00
        BEST RMSE:    622.78
        BEST MAPE:    4046.19
        BEST R^2:     0.307

tlc:
        CV R^2:       [0.38641446 0.38374875 0.37846488 0.39794431 0.38757503]                  

        results of model predictions on all training data:
        BEST MAE:     1002.05
        BEST RMSE:    1395.95
        BEST MAPE:    4921.56
        BEST R^2:     0.562

